# 5a Compare Proposals Original

This notebook runs the pooled Human vs All-AI proposal analyses on the original-text branch.

It reuses the prepared outputs from `4a_prepare_proposal_for_analysis.ipynb` and the human review-score overlays from `4b_prepare_review_for_analysis.ipynb`, then saves per-condition and cross-condition pooled results.

In [ ]:
CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSION = 'original'

N_BOOT = 1000
BOOT_SEED = 42
N_SUBSAMPLE = 23
N_PERM = 10000
STYLE_PERM = 1000
GRID_BINS = 8


In [ ]:
import json
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from proposal_generation import find_project_root
from proposal_comparison import (
    PRIMARY_DIVERSITY_METRICS,
    build_proposal_metrics_master,
    compute_literature_self_knn_cache,
    cross_condition_summary_table,
    generate_or_load_bootstrap_samples,
    get_group_indices,
    load_condition_analysis_inputs,
    permutation_test_group_metric,
    pooled_bootstrap_metric_distribution,
    run_simple_cluster_analysis,
    run_simple_topic_analysis,
    simple_style_features,
    style_classifier_permutation,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
sns.set_theme(style='whitegrid', context='talk')
LITERATURE_EMBEDDINGS_PATH = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'relevant_literature_embeddings.pkl'
print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions to run: {CONDITIONS_TO_RUN}')


In [ ]:
condition_results = {}
cross_condition_rows = []

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== 5a pooled proposal comparison: {condition} ===')
    analysis = load_condition_analysis_inputs(PROJECT_ROOT, condition, text_version=TEXT_VERSION)
    group_idx = get_group_indices(analysis.proposal_master)
    human_idx = group_idx['Human']
    ai_idx = group_idx['All AI']

    if len(human_idx) != 23:
        raise RuntimeError(f'{condition}: expected 23 Human proposals, found {len(human_idx)}')
    if len(ai_idx) != 69:
        raise RuntimeError(f'{condition}: expected 69 pooled AI proposals, found {len(ai_idx)}')

    tables_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'proposals' / TEXT_VERSION / 'all_ai'
    figures_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'proposals' / TEXT_VERSION / 'all_ai'
    shared_cache_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'proposals' / TEXT_VERSION / 'shared_cache'
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    shared_cache_dir.mkdir(parents=True, exist_ok=True)

    bootstrap_path = shared_cache_dir / f'bootstrap_ai_idx_samples_n{N_SUBSAMPLE}_seed{BOOT_SEED}.npy'
    bootstrap_ai_idx_samples = generate_or_load_bootstrap_samples(
        ai_idx,
        output_path=bootstrap_path,
        n_boot=N_BOOT,
        n_subsample=N_SUBSAMPLE,
        seed=BOOT_SEED,
    )

    lit_self_knn_path = shared_cache_dir / 'lit_knn_distances_50.npy'
    lit_knn_distances_50 = compute_literature_self_knn_cache(
        LITERATURE_EMBEDDINGS_PATH,
        output_path=lit_self_knn_path,
        k=50,
    )
    lit_mean_knn_10 = lit_knn_distances_50[:, :10].mean(axis=1)

    proposal_metrics_master = build_proposal_metrics_master(analysis, lit_self_knn_mean10=lit_mean_knn_10)
    proposal_metrics_master.to_csv(tables_dir / 'proposal_metrics_master.csv', index=False)

    diversity_rows = []
    bootstrap_frames = []
    for metric_name in PRIMARY_DIVERSITY_METRICS + ['mst_dispersion']:
        perm = permutation_test_group_metric(
            analysis.pairwise_full,
            human_idx,
            ai_idx,
            metric_name=metric_name,
            n_perm=N_PERM,
            seed=BOOT_SEED,
            coords=analysis.umap2d,
        )
        boot_df = pooled_bootstrap_metric_distribution(
            analysis.pairwise_full,
            human_idx,
            bootstrap_ai_idx_samples,
            metric_name=metric_name,
            coords=analysis.umap2d,
        )
        human_value = float(boot_df['human_value'].iloc[0])
        ai_values = boot_df['ai_value'].to_numpy()
        diversity_rows.append(
            {
                'condition': condition,
                'metric': metric_name,
                'human_value': human_value,
                'ai_boot_mean': float(np.mean(ai_values)),
                'ai_boot_sd': float(np.std(ai_values)),
                'effect_human_minus_ai_boot_mean': float(human_value - np.mean(ai_values)),
                'permutation_p_value': perm['permutation_p_value'],
                'inference_primary': 'permutation',
            }
        )
        bootstrap_frames.append(boot_df.assign(condition=condition))
        cross_condition_rows.append(
            {
                'condition': condition,
                'metric': metric_name,
                'effect_human_minus_ai': float(human_value - np.mean(ai_values)),
                'permutation_p_value': perm['permutation_p_value'],
                'analysis_family': 'proposal_space_diversity',
            }
        )

    diversity_summary_df = pd.DataFrame(diversity_rows)
    diversity_summary_df.to_csv(tables_dir / 'diversity_summary_human_vs_allai.csv', index=False)
    bootstrap_metrics_df = pd.concat(bootstrap_frames, ignore_index=True)
    bootstrap_metrics_df.to_csv(tables_dir / 'proposal_metrics_summary_human_vs_allai.csv', index=False)

    metric_corr_df = proposal_metrics_master[
        ['mean_pairwise_distance', 'nearest_neighbor_distance', 'literature_element_novelty_k1', 'literature_mean_knn_novelty_k10', 'literature_local_density_normalized_novelty']
    ].corr(numeric_only=True)
    metric_corr_df.to_csv(tables_dir / 'proposal_diversity_metric_correlations.csv')

    topic_df, topic_summary = run_simple_topic_analysis(analysis.proposal_master)
    topic_df.to_csv(tables_dir / 'topic_assignments_human_vs_allai.csv', index=False)
    with open(tables_dir / 'topic_distribution_summary.json', 'w') as f:
        json.dump(topic_summary, f, indent=2)

    cluster_df, cluster_summary = run_simple_cluster_analysis(
        np.asarray(analysis.full_embeddings['embeddings']),
        analysis.proposal_master,
    )
    cluster_df.to_csv(tables_dir / 'cluster_assignments_human_vs_allai.csv', index=False)
    with open(tables_dir / 'cluster_segregation_summary.json', 'w') as f:
        json.dump(cluster_summary, f, indent=2)

    style_X = simple_style_features(analysis.proposal_master['full_text'])
    style_y = (analysis.proposal_master['source_type'] == 'ai').astype(int).to_numpy()
    style_result = style_classifier_permutation(style_X, style_y, n_perm=STYLE_PERM, seed=BOOT_SEED)
    style_feature_summary = pd.DataFrame({
        'feature': style_X.columns,
        'human_mean': style_X.loc[analysis.proposal_master['source_type'] == 'human'].mean().to_numpy(),
        'ai_mean': style_X.loc[analysis.proposal_master['source_type'] == 'ai'].mean().to_numpy(),
    })
    style_feature_summary.to_csv(tables_dir / 'style_feature_contrasts_human_vs_allai.csv', index=False)
    with open(tables_dir / 'style_classifier_summary.json', 'w') as f:
        json.dump(style_result, f, indent=2)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=diversity_summary_df, x='metric', y='effect_human_minus_ai_boot_mean', ax=ax, color='#4A90E2')
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'{condition}: Human - All AI pooled diversity effect')
    ax.set_ylabel('Effect (Human minus AI bootstrap mean)')
    ax.set_xlabel('Metric')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    fig.savefig(figures_dir / 'diversity_effects_human_vs_allai.png', dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_df = analysis.proposal_master.copy()
    plot_df['umap_x'] = analysis.umap2d[:, 0]
    plot_df['umap_y'] = analysis.umap2d[:, 1]
    sns.scatterplot(data=plot_df, x='umap_x', y='umap_y', hue='source_type', style='source_type', ax=ax)
    ax.set_title(f'{condition}: proposal-space UMAP (original)')
    plt.tight_layout()
    fig.savefig(figures_dir / 'proposal_space_umap_human_vs_allai.png', dpi=200)
    plt.close(fig)

    condition_results[condition] = {
        'analysis': analysis,
        'diversity_summary_df': diversity_summary_df,
        'proposal_metrics_master': proposal_metrics_master,
        'topic_summary': topic_summary,
        'cluster_summary': cluster_summary,
        'style_result': style_result,
        'tables_dir': tables_dir,
        'figures_dir': figures_dir,
        'shared_cache_dir': shared_cache_dir,
    }
    print(f'Finished pooled original-text proposal analysis for {condition}')


In [ ]:
cross_tables_dir = PROJECT_ROOT / 'results' / 'tables' / 'proposals' / TEXT_VERSION / 'cross_condition'
cross_figures_dir = PROJECT_ROOT / 'results' / 'figures' / 'proposals' / TEXT_VERSION / 'cross_condition'
cross_tables_dir.mkdir(parents=True, exist_ok=True)
cross_figures_dir.mkdir(parents=True, exist_ok=True)

cross_condition_df = cross_condition_summary_table(cross_condition_rows)
cross_condition_df.to_csv(cross_tables_dir / 'pooled_human_vs_allai_cross_condition_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cross_condition_df[cross_condition_df['metric'].isin(PRIMARY_DIVERSITY_METRICS)], x='condition', y='effect_human_minus_ai', hue='metric', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Cross-condition primary diversity effects (original proposals)')
ax.set_ylabel('Effect (Human minus AI)')
ax.set_xlabel('Condition')
plt.tight_layout()
fig.savefig(cross_figures_dir / 'primary_diversity_effects_cross_condition.png', dpi=200)
plt.close(fig)

cross_condition_df
